# Weld-YOLO11-SO: Dual Environment (Local & Google Colab) Training Notebook

This notebook is fully configured for **seamless execution BOTH locally** (inside `main_tranning/` subfolder) **AND on Google Colab** (with GPU acceleration).

### Weld-YOLO11-SO Highlights:
- **WaveletBlock**: Multiscale frequency domain feature enhancement
- **WeldSimAM**: Directional parameter-free attention for fine weld seams
- **DySample**: Dynamic lightweight upsampling
- **AHSFPN**: Adaptive Hierarchical Scale Feature Pyramid Network
- **4 Detection Scales**: P2 (stride 4) to P5 (stride 32) for small defect detection


In [3]:
# 1. Environment & Path Resolution (Auto-detects Colab vs Local subfolder)
import os
import sys
import pathlib
import subprocess

# Prevent OpenMP runtime conflict on Windows
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

# Auto-install dependencies if missing
try:
    import ultralytics
    import PyWavelets
    import albumentations
except ModuleNotFoundError:
    print("📦 Installing required packages (ultralytics, PyWavelets, opencv-python, albumentations, kagglehub)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics", "PyWavelets", "opencv-python", "albumentations", "kagglehub"])

# Check if running on Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🌐 Environment: Google Colab detected")
    repo_url = "https://github.com/abhisekkundu-DS/Weld-YOLO11-SO"
    repo_name = "Weld-YOLO11-SO"
    if not os.path.exists(f"/content/{repo_name}"):
        os.system(f"git clone {repo_url} /content/{repo_name}")
    os.chdir(f"/content/{repo_name}")
    project_root = pathlib.Path(f"/content/{repo_name}").resolve()
else:
    print("💻 Environment: Local Workspace detected")
    cwd = pathlib.Path('.').resolve()
    # Auto-resolve parent project root if executed inside main_tranning/ subfolder
    if (cwd / "models" / "weld_yolo11.yaml").exists():
        project_root = cwd
    elif (cwd.parent / "models" / "weld_yolo11.yaml").exists():
        project_root = cwd.parent
        os.chdir(project_root)  # Set CWD to project root
    else:
        project_root = cwd

# Add project root to sys.path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 2. Register Weld-YOLO11-SO Custom Modules with Ultralytics
from models.modules import register_custom_modules
register_custom_modules()

import torch
print(f"\n✅ Active Working Directory: {os.getcwd()}")
print(f"   PyTorch CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU Device: {torch.cuda.get_device_name(0)}")


📦 Installing required packages (ultralytics, PyWavelets, opencv-python, albumentations, kagglehub)...
🌐 Environment: Google Colab detected
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Successfully registered Weld-YOLO11-SO custom modules with Ultralytics framework.

✅ Active Working Directory: /content/Weld-YOLO11-SO
   PyTorch CUDA Available: False


In [4]:
# Verify Weld-YOLO11-SO Model Architecture
from ultralytics import YOLO

model_yaml = project_root / "models" / "weld_yolo11.yaml"
print(f"Loading model architecture from: {model_yaml}")

model = YOLO(str(model_yaml), task="detect")
print("\n✅ Weld-YOLO11-SO model architecture loaded and verified successfully!")


Loading model architecture from: /content/Weld-YOLO11-SO/models/weld_yolo11.yaml
WARNING ⚠️ no model scale passed. Assuming scale='n'.


RuntimeError: Given groups=128, weight of size [384, 3, 3, 3], expected input[1, 96, 32, 32] to have 384 channels, but got 96 channels instead

In [ ]:
# Locate or Download Welding Defect Dataset via KaggleHub
from pathlib import Path
import kagglehub

try:
    dataset_path = Path(kagglehub.dataset_download("sukmaadhiwijaya/welding-defect-object-detection"))
    print(f"✅ Kaggle Welding Dataset ready at: {dataset_path}")
except Exception as e:
    print(f"Kaggle download fallback: {e}")
    dataset_path = project_root / "welding_dataset"


In [ ]:
# Augmentation Pipeline tailored for Welding Defect Detection
import cv2
import albumentations as A

def create_augmentation_pipeline():
    return A.Compose([
        A.HorizontalFlip(p=0.50),
        A.Rotate(limit=10, border_mode=cv2.BORDER_REFLECT_101, p=0.50),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.50),
        A.Affine(scale=(0.90, 1.10), translate_percent=(-0.05, 0.05), border_mode=cv2.BORDER_REFLECT_101, p=0.50),
        A.GaussianBlur(blur_limit=(3, 3), p=0.10)
    ], bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"], min_visibility=0.20))

print("✅ Albumentations pipeline initialized.")


In [ ]:
# Dataset Configuration Setup
output_dir = project_root / "Welding_Augmented"
for split in ["train", "valid", "test"]:
    (output_dir / split / "images").mkdir(parents=True, exist_ok=True)
    (output_dir / split / "labels").mkdir(parents=True, exist_ok=True)

data_yaml_content = f"""path: {output_dir.absolute()}
train: train/images
val: valid/images
test: test/images

nc: 3
names:
  0: Bad Weld
  1: Good Weld
  2: Defect
"""
data_yaml = output_dir / "data.yaml"
data_yaml.write_text(data_yaml_content)
print(f"✅ Dataset configuration written to: {data_yaml}")


In [ ]:
# Train Weld-YOLO11-SO Model
from ultralytics import YOLO

train_data_yaml = project_root / "Welding_Augmented" / "data.yaml"

# If augmented dataset is empty locally, fallback to 100-sample MNIST benchmark for fast local testing
has_images = any((output_dir / "train" / "images").iterdir())
if not has_images:
    print("💡 Local small test mode: Generating 100-sample dataset for model validation...")
    from train_mnist import prepare_mnist_yolo_dataset
    train_data_yaml = prepare_mnist_yolo_dataset(project_root / "mnist_dataset", num_samples=100)

print(f"🚀 Starting training with data config: {train_data_yaml}")
model = YOLO(str(project_root / "models" / "weld_yolo11.yaml"), task="detect")

results = model.train(
    data=str(train_data_yaml),
    imgsz=1024,
    epochs=3 if not IN_COLAB else 20,  # 3 fast test epochs locally, 20 epochs in Colab
    batch=2,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=0 if os.name == 'nt' else 2,
    amp=True,
    project="runs/weld_training",
    name="weld_yolo11_so",
    exist_ok=True
)

print("\n✅ Training Run Completed Successfully!")


In [ ]:
# Evaluate Metrics
metrics = model.val()
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"mAP50:    {metrics.box.map50:.4f}")
